# 01 . Bronze - Raw Event Ingestion


Ingest `payments_events.json` into `novalake.bronze.raw_events`, preserving the source representation faithfully. No restructuring, no type fixes - that's Silver's job.

In [0]:
%python

from pyspark.sql import functions as F

catalog = "novalake"
raw_path = f"/Volumes/{catalog}/bronze/landing/payments_events.json"

In [0]:
df_inferred = spark.read.json(raw_path)
print(f"Rows: {df_inferred.count()}")
print(f"Top-level column: {len(df_inferred.columns)} -> {df_inferred.columns}")
df_inferred.printSchema()

In [0]:
df_inferred.select("payload.risk").printSchema()


## Checking what permissive mode actually catches
Spark's JSON reader defaults to PERMISSIVE and auto-adds `_corrupt_record` for syntactically broken lines, when inference finds any, Our mess is type drift across valid records - different thing entirely, and not what this column protects against.

In [0]:
if "_corrupt_record" in df_inferred.columns:
    n = df_inferred.filter(F.col("_corrupt_record").isNotNull()).count()
    print(f"Corrupt records: {n}")
else:
    print("No '_corrupt_record' column at all - Spark found zero syntactically broken lines")

In [0]:

bronze_df = (
    df_inferred
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.bronze.raw_events"))

In [0]:
display(spark.sql(f"SELECT event_type, schema_version, count(*) as n FROM novalake.bronze.raw_events GROUP BY event_type, schema_version ORDER BY event_type, schema_version"))

In [0]:
df_inferred.select("event_timestamp").printSchema()